# 02 — Modelling strategy  [METHODS]

Why OLS is inadequate, the NB specification, predictor selection, and the selection procedure's own instability. Consolidated from `archive/supervisor_run_2026-08-16.ipynb` and `archive/rate_total_analysis.ipynb`.

## Contents

- **2.1** OLS → WLS → NB ladder
- **2.2** Diagnostics, correctly specified (re-run)
- **2.3** Overdispersion and the NB specification with offset
- **2.4** Correlation redundancy screen
- **2.5** LOWESS functional-form panels
- **2.6** Selection procedure and its instability

### 2.1 OLS → WLS → NB ladder  [METHODS — SETTLED]

Four baseline fits on the fixed 5-predictor set (OLS, WLS, Binomial GLM, Binomial GLM/HC1), reference-only — nothing here feeds selection or is cited as a finding downstream.

- **OLS fitted outside [0,1]:** 0/299 — this dataset happens not to expose OLS's floor/ceiling problem, but the model still has no mechanism preventing it.
- **GLM dispersion (binomial):** 9.75 — strongly overdispersed relative to binomial variance, consistent with the NB choice used everywhere else.
- **Largest OLS→WLS coefficient shift:** `log_population_density_c`, **94.2%** — the clearest demonstration of the cost of ignoring unequal `social_reviews` denominators across constituencies. This remains the headline number for this section.

*(Source: `outputs/R1_summary_col_baseline.txt`, `archive/supervisor_run_2026-08-16.ipynb` cell `R1`. Binomial-GLM and binomial-HC1 rows kept here for the OLS→WLS comparison only — not treated as candidate models anywhere downstream.)*

### 2.2 Diagnostics, correctly specified [METHODS — SETTLED, re-run]

The version previously reported here (Task A / R1, `analysis_checks_summary.md`) tested the wrong thing twice: it correlated the raw OLS residual (not squared) with `social_reviews`, which is a **mean-trend** test, not a variance test; and its Breusch–Pagan used only the model predictors in the auxiliary regression, making denominator-driven variance structurally invisible to it. Re-run below with the corrected specifications.

In [ ]:
# Self-contained setup: rebuild analysis_df and the 5-predictor OLS baseline
# (same construction as archive/supervisor_run_2026-08-16.ipynb cells 2-7, 61)
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings("ignore", category=FutureWarning)

ANALYSIS_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026")
GMAPS_DIR = ANALYSIS_DIR / "gmaps_data"
SOCIO_DIR = ANALYSIS_DIR / "socio-economic_data"
SHAPEFILE_PATH = Path("/home/ethel/Desktop/swimming_pool_project/btw25_geometrie_wahlkreise_shp/btw25_geometrie_wahlkreise_shp.shp")

corpus = pd.read_parquet(GMAPS_DIR / "master_corpus_final.parquet",
                          columns=["review_id", "unique_identifier", "social", "racialised",
                                   "total_text_reviews", "latitude", "longitude", "place_name"])
political = pd.read_csv(SOCIO_DIR / "federal_baseline_eng_selected_political.csv")
socio_political = pd.read_csv(SOCIO_DIR / "socio_political_data.csv")

places = corpus[["unique_identifier", "place_name", "latitude", "longitude"]].drop_duplicates(subset="unique_identifier").reset_index(drop=True)
places_geo = gpd.GeoDataFrame(places, geometry=gpd.points_from_xy(places["longitude"], places["latitude"]), crs="EPSG:4326").to_crs("EPSG:25832")
constituencies = gpd.read_file(SHAPEFILE_PATH)
if (~constituencies.geometry.is_valid).any():
    constituencies["geometry"] = constituencies.geometry.buffer(0)
germany_boundary = constituencies.dissolve().geometry.iloc[0]
germany_gdf = gpd.GeoDataFrame(geometry=[germany_boundary], crs=constituencies.crs)
inside_idx = gpd.sjoin(places_geo, germany_gdf, predicate="within", how="inner").index
places_inside = places_geo.loc[places_geo.index.isin(inside_idx)].copy()
joined = gpd.sjoin_nearest(places_inside, constituencies[["WKR_NR", "WKR_NAME", "LAND_NR", "LAND_NAME", "geometry"]], how="left")
joined = joined.drop_duplicates(subset="unique_identifier", keep="first")
place_constituency = joined.rename(columns={"WKR_NR": "constituency_code", "WKR_NAME": "constituency", "LAND_NAME": "state"})[
    ["unique_identifier", "constituency_code", "constituency", "state"]].reset_index(drop=True)
corpus = corpus.merge(place_constituency, on="unique_identifier", how="left")

backbone = political[["state", "constituency_code", "constituency"]].copy()
agg = (corpus.groupby("constituency_code")
       .agg(pool_count=("unique_identifier", "nunique"),
            total_text_reviews=("review_id", "count"),
            social_reviews=("social", lambda s: (s == "yes").sum()),
            racialised_reviews=("racialised", lambda s: (s == "yes").sum()))
       .reset_index())
const_agg = backbone.merge(agg, on="constituency_code", how="left")
for c in ["pool_count", "total_text_reviews", "social_reviews", "racialised_reviews"]:
    const_agg[c] = const_agg[c].fillna(0).astype(int)

analysis_df = const_agg.merge(socio_political, on=["state", "constituency_code", "constituency"], how="left")
analysis_df["log_population_density"] = np.log(analysis_df["population_density_per_km2"])
EAST_STATES = ["Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen"]
analysis_df["is_east"] = analysis_df["state"].isin(EAST_STATES).astype(int)
CHECK_COLS = ["unemployment_total_pct", "sgb_ii_recipients_per_1k", "school_grads_higher_education_entrance_pct",
              "age_60_74_pct", "age_75_plus_pct"]
analysis_df["disposable_income_1k"] = analysis_df["disposable_income_eur_per_inhabitant"] / 1000
dup_groups = analysis_df.groupby(CHECK_COLS + ["disposable_income_1k"])["constituency"].apply(list)
dup_groups = dup_groups[dup_groups.apply(len) > 1]
flagged = {c for consts in dup_groups for c in consts}
analysis_df["in_split_city"] = analysis_df["constituency"].isin(flagged).astype(int)
for c in ["foreign_population_pct", "log_population_density", "sgb_ii_recipients_per_1k"]:
    analysis_df[c + "_c"] = analysis_df[c] - analysis_df[c].mean()

FULL_FORMULA = "racialised_reviews ~ foreign_population_pct_c + log_population_density_c + sgb_ii_recipients_per_1k_c + is_east + in_split_city"
preds = FULL_FORMULA.split("~")[1].strip()
df_r1 = analysis_df.copy()
df_r1["rate_social"] = df_r1["racialised_reviews"] / df_r1["social_reviews"]
ols = smf.ols(f"rate_social ~ {preds}", data=df_r1).fit()
print(f"analysis_df: {analysis_df.shape}, OLS fit on rate_social, n={ols.nobs:.0f}")

In [ ]:
# 2.2 -- heteroskedasticity, correctly specified
from statsmodels.stats.diagnostic import het_breuschpagan, het_white

resid = ols.resid
resid_sq = resid ** 2
log_n = np.log(df_r1["social_reviews"])

# corrected funnel check: does *variance* (not mean) shrink as the denominator grows?
X = sm.add_constant(log_n)
funnel_model = sm.OLS(resid_sq, X).fit()
slope, slope_p = funnel_model.params["social_reviews"], funnel_model.pvalues["social_reviews"]
print(f"corrected funnel check -- residual**2 ~ log(social_reviews): slope={slope:.6f}, p={slope_p:.4f}")

# Breusch-Pagan with log(social_reviews) added to the auxiliary regression
exog_plus = np.column_stack([ols.model.exog, log_n.values])
bp_stat, bp_p, _, _ = het_breuschpagan(resid, exog_plus)
print(f"Breusch-Pagan (predictors + log(social_reviews)): LM={bp_stat:.3f}, p={bp_p:.4f}")

# White's test as a cross-check (squares + cross-products of predictors)
white_stat, white_p, _, _ = het_white(resid, ols.model.exog)
print(f"White's test (predictors, squares, cross-products): LM={white_stat:.3f}, p={white_p:.4f}")

verdict = "no evidence of heteroskedasticity" if (slope_p >= 0.05 and bp_p >= 0.05 and white_p >= 0.05) else "heteroskedasticity detected by at least one corrected test"
print(f"\nVerdict: {verdict}")

**Result:** all three corrected tests agree with the original (flawed) conclusion — no evidence of heteroskedasticity in the OLS residuals (funnel-check slope p=0.94, Breusch-Pagan p=0.38, White's p=0.77). The original check happened to land on the right answer for the wrong reason; the fix doesn't change the conclusion, but the earlier version couldn't have detected denominator-driven variance even if it had been present, so it wasn't evidence either way until now.

### 2.3 Overdispersion and the NB specification with offset  [METHODS — SETTLED]

Poisson dispersion (Pearson χ²/df) = 9.60 — strongly overdispersed. Negative binomial with `offset=log(social_reviews)` brings dispersion to **0.826** (α̂=0.375, moment-estimated from the Poisson fit) — a well-specified count model for the 5-predictor set. Analysis-of-deviance confirms each predictor's addition is a significant improvement in both the Poisson and NB versions. *(Source: `outputs/D3_three_model_comparison.csv`, `outputs/D4_deviance_table_*.csv`, `archive/supervisor_run_2026-08-16.ipynb` cells `D3`/`D4`.)*

### 2.4 Correlation redundancy screen  [METHODS — SETTLED]

Step 1–2 of predictor selection (drop one of any pair of predictors correlated above threshold) is mathematically DV-independent — it necessarily produces the same drops regardless of which outcome (`rate_social` or `rate_total`) is being modelled: `unemployment_total_pct` and `second_vote_greens_pct` are dropped for redundancy with predictors already in the final 5-predictor set. *(Source: `rate_total_analysis.ipynb` C3a, moved to `archive/`.)*

### 2.5 LOWESS functional-form panels  [METHODS — SETTLED]

West/East LOWESS panels against `rate_total` (`outputs/rate_total/C3b_lowess_shape_panels.png`, shape calls automated as slope-sign agreement + linear R² rather than eyeballed — flagged as a methodological substitution, worth a visual sanity check before treating the calls as final). The five predictors in the final model show no shape problems in either region. Two out-of-model observations worth carrying into the discussion: `sgb_ii_recipients_per_1k` diverges above ~125 where East has no data, and `second_vote_afd_pct` slopes **downward** in both West and East — consistent with §4.2's note that the naive AfD hypothesis runs the wrong direction. `second_vote_left_bsw`, `disposable_income`, and `turnout_pct` are the clearly nonlinear predictors and are not in the final model.

**Open note:** a rate_social-specific LOWESS panel (as opposed to the rate_total panel above) was not located as a separate saved artifact during this consolidation — see `../ANALYSIS_LOG.md` Part III. Regenerating it would be a new analysis, out of scope for this pass; flagged rather than run.

### 2.6 Selection procedure and its instability  [METHODS — SETTLED]

Sequential significance-pruning is unstable and can make the fit *worse*. Under `rate_total`, stepwise pruning retained only `foreign_population_pct_c`, but the full 5-predictor model has 4/5 CIs excluding zero **and** a lower AIC (2287.8 vs. 2300.8 for the pruned model). The pruned model's `foreign_population` coefficient (0.0546) is inflated relative to the full model's (0.0356) because it absorbs the variance the dropped predictors would otherwise have explained. Framed as a limitation of the selection procedure itself, not a finding about the outcome — the final model used throughout notebook 03 is the full 5-predictor specification, not the pruned one. *(Source: `rate_total_analysis.ipynb` C3c/C3d, `analysis_checks_summary.md` Task C.)*